# Exercise 3 — Neural-network model and MPC by enumeration

We **replace the physical model with a neural network** learned from data,

$$\hat T_{k+1} = f_{NN}(T_k,\ u_k,\ T_{o,k}),$$

and use it for predictive control. The heating power can only take five values, $u \in \{0, 500, 1000, 1500, 2000\}$ W. With a horizon of $H$ steps there are $5^H$ input sequences. We **try all of them**, compute

$$J = \sum_{i=1}^{H} (r - \hat T_i)^2 + \lambda \sum_{i=0}^{H-1} u_i^2, \qquad 19 \le \hat T_i \le 24\ ^\circ\text{C},$$

and apply the first input of the best feasible sequence.

The 1R1C model of Exercises 1–2 plays the **real building**: it generates the training data and is the plant in the simulation. The controller only knows the neural network.

In [ ]:
# Google Colab only: download the course files and check the packages.
# When you run the notebook on your own computer, this cell does nothing.
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("building_model.py"):
        if not os.path.exists("athens_neural_network_for_control"):
            subprocess.run(["git", "clone", "-q", "https://github.com/li-yun/athens_neural_network_for_control.git"], check=True)
        os.chdir("athens_neural_network_for_control")
    r = subprocess.run([sys.executable, "install_packages.py"], capture_output=True, text=True)
    out = r.stdout + r.stderr
    print(out[out.find("2/3"):] if r.returncode == 0 else out)   # show the check results
    if r.returncode != 0:
        raise RuntimeError("Setup failed, see the messages above.")

In [ ]:
import time
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from building_model import (FirstOrderBuilding, outdoor_temperature, generate_training_data,
                            plot_closed_loop, evaluate)

building = FirstOrderBuilding()          # the "real" building
r, T0 = 21.0, 20.0
levels = np.array([0.0, 500.0, 1000.0, 1500.0, 2000.0])
To = outdoor_temperature('winter')       # same winter profile as before (measured To - 12 degC)

## 1. Training data

Random heating power and random initial temperatures on the winter outdoor temperature; each sample is $(T_k, u_k, T_{o,k}) \rightarrow T_{k+1}$.

In [ ]:
data_train = generate_training_data(building, n_episodes=60, seed=0)
data_test = generate_training_data(building, n_episodes=15, seed=1)

X_train, y_train = data_train[['T', 'u', 'To']].values, data_train['T_next'].values
X_test, y_test = data_test[['T', 'u', 'To']].values, data_test['T_next'].values
data_train.head()

## 2. Task: train the neural network

Create a `StandardScaler` followed by an `MLPRegressor`, and fit it on `X_train, y_train`. Use one hidden layer with 16 neurons, `activation='tanh'`, `max_iter=3000, tol=1e-7, n_iter_no_change=50, random_state=0`.

*Hint:* `make_pipeline(StandardScaler(), MLPRegressor(...))`

In [ ]:
# TODO: create the network and fit it (2 lines)
nn = ...


In [ ]:
def nn_step(T, u, To):
    """Neural-network model: predict T[k+1] for arrays (or scalars) T, u, To."""
    T, u, To = np.broadcast_arrays(np.atleast_1d(T), np.atleast_1d(u), np.atleast_1d(To))
    return nn.predict(np.column_stack([T, u, To]))

# Validation 1: one-step prediction on the test set
print(f'one-step RMSE: {np.sqrt(np.mean((nn.predict(X_test) - y_test)**2)):.3f} K  (noise = 0.02 K)')

# Validation 2: multi-step simulation over 90 h, network fed with its own predictions
u_val = np.clip(1000 * (21 - building.simulate(20, 700, To)), 0, 2000)
T_true = building.simulate(20, u_val, To)
T_nn = [20.0]
for k in range(len(To) - 1):
    T_nn.append(nn_step(T_nn[-1], u_val[k], To[k])[0])

t = np.arange(len(To)) / 2
plt.figure(figsize=(10, 3))
plt.plot(t, T_true, label='real building (1R1C)'); plt.plot(t, T_nn, '--', label='neural network')
plt.xlabel('Time [h]'); plt.ylabel('T [°C]'); plt.legend(); plt.grid(alpha=0.3); plt.show()

## 3. Task: MPC by enumeration

`self.U` already holds all $5^H$ sequences, one per row. All sequences are predicted **at once** (vectorised): `T` is an array with one predicted temperature per sequence. Complete the three lines inside the loop.

In [ ]:
class EnumerationMPC:
    """MPC that tries every input sequence in levels^H. Call: controller(r_k, T_k, k) -> u_k."""

    def __init__(self, step, H, lam, To_forecast, T_min=19.0, T_max=24.0):
        self.step = step                  # prediction model: T_next = step(T, u, To), works on arrays
        self.H, self.lam, self.To_forecast = H, lam, To_forecast
        self.T_min, self.T_max = T_min, T_max
        self.U = np.array(list(itertools.product(levels, repeat=H)))   # all sequences, shape (5**H, H)
        self.solve_times = []

    def __call__(self, r_k, T_k, k):
        t0 = time.perf_counter()
        U = self.U
        n = len(U)                        # number of sequences
        T = np.full(n, T_k)               # predicted temperature of every sequence, starts at T_k
        J = np.zeros(n)                   # cost of every sequence
        feasible = np.ones(n, dtype=bool)

        for i in range(self.H):
            To_i = np.full(n, self.To_forecast[k + i])
            # TODO: 3 lines
            #   1. predict the next temperature of all sequences:  T = self.step(T, U[:, i], To_i)
            #   2. add the cost of step i:   (r_k - T)^2 + lam * U[:, i]^2
            #   3. update feasible: the temperature must stay in [T_min, T_max]
            ...

        J[~feasible] = np.inf             # discard sequences that leave the comfort band
        best = np.argmin(J)
        self.solve_times.append(time.perf_counter() - t0)
        return U[best, 0]                 # apply only the first input of the best sequence

## 4. Closed loop: neural-network model vs. true model

The same controller is run once with the neural network and once with the true model, so the difference shows the effect of the model error.

In [ ]:
H, lam = 4, 1e-7                                        # 4 x 30 min = 2 h -> 625 sequences
To_forecast = np.concatenate([To, np.full(H, To[-1])])

mpc_nn = EnumerationMPC(nn_step, H, lam, To_forecast)
mpc_true = EnumerationMPC(building.step, H, lam, To_forecast)
res_nn = building.simulate_closed_loop(T0, r, To, controller=mpc_nn, pass_k=True)
res_true = building.simulate_closed_loop(T0, r, To, controller=mpc_true, pass_k=True)

plot_closed_loop([res_nn, res_true], labels=['MPC with NN model', 'MPC with true model'],
                 u_max=2000, T_band=(19, 24))
plt.show()
pd.DataFrame({'NN model': {**evaluate(res_nn, T_min=19, T_max=24), 'time per step [ms]': 1e3 * np.mean(mpc_nn.solve_times)},
              'true model': {**evaluate(res_true, T_min=19, T_max=24), 'time per step [ms]': 1e3 * np.mean(mpc_true.solve_times)}}).T.round(3)

## 5. Computation time vs. horizon

The number of sequences is $5^H$.

In [ ]:
Hs, t_step = range(1, 9), []
for Hh in Hs:
    ctrl = EnumerationMPC(nn_step, Hh, lam, np.concatenate([To, np.full(Hh, To[-1])]))
    for k in range(2):
        ctrl(r, T0, k)
    t_step.append(1e3 * np.mean(ctrl.solve_times))
    print(f'H = {Hh}: {5**Hh:7d} sequences, {t_step[-1]:7.1f} ms per step')

plt.figure(figsize=(6, 3.5))
plt.semilogy(list(Hs), t_step, 'o-')
plt.xlabel('horizon H'); plt.ylabel('time per step [ms]'); plt.grid(alpha=0.3, which='both'); plt.show()